<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z343_MundoIdealEscalado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mundo Ideal Escalado — benchmark sintético de normalizaciones

**Pregunta**: si dos productos tienen la *misma forma* pero distinta escala (uno vende 10x más que el otro),
¿qué normalización logra que se vean como la misma serie?

## Diseño del experimento

1. **Shapes base**: 8 formas de 36 meses (tendencia+, tendencia−, estable, estacional, spike, V, escalón, ruido)
2. **Variantes por shape**: K=12 series con escala aleatoria log-uniforme en [0.05, 200]
3. **Perturbaciones realistas**:
   - arranque tardío: primeros N meses = −1 (producto no existía)
   - ceros aleatorios en el medio (existía pero no vendió)
4. **Normalizaciones**: max, l2, index, reciente (media últimos 3m), + sin normalizar
5. **Evaluación**:
   - distancia coseno intra-shape vs inter-shape
   - clustering jerárquico → ¿agrupa bien?
   - árbol de decisión: ¿puede separar shapes con features normalizadas?
6. **Desescalado**: predecir t+2 normalizado → × escala → error vs original

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial.distance import cosine
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import adjusted_rand_score
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
T = 36       # meses por serie
K = 12       # variantes por shape
HORIZONTE = 2
print('OK')

# 1. Shapes base

In [ ]:
t = np.arange(T)

def shape_tendencia_pos():
    return 1 + t / T * 3 + np.random.normal(0, 0.05, T)

def shape_tendencia_neg():
    return 4 - t / T * 3 + np.random.normal(0, 0.05, T)

def shape_estable():
    return np.ones(T) * 2 + np.random.normal(0, 0.08, T)

def shape_estacional():
    return 1.5 + np.sin(2 * np.pi * t / 12) + 0.5 * np.sin(2 * np.pi * t / 6) + np.random.normal(0, 0.05, T)

def shape_spike():
    s = np.ones(T) * 1.2 + np.random.normal(0, 0.05, T)
    s[T // 3] += 8.0
    s[2 * T // 3] += 5.0
    return s

def shape_v():
    caida = np.linspace(3, 0.5, T // 2)
    subida = np.linspace(0.5, 3, T - T // 2)
    return np.concatenate([caida, subida]) + np.random.normal(0, 0.05, T)

def shape_escalon():
    s = np.ones(T) * 1.0
    s[T // 2:] = 4.0
    return s + np.random.normal(0, 0.05, T)

def shape_ruido():
    return np.abs(np.random.normal(2, 1.5, T))

SHAPES = {
    'tendencia+': shape_tendencia_pos,
    'tendencia-': shape_tendencia_neg,
    'estable':    shape_estable,
    'estacional': shape_estacional,
    'spike':      shape_spike,
    'V':          shape_v,
    'escalon':    shape_escalon,
    'ruido':      shape_ruido,
}
N_SHAPES = len(SHAPES)
print(f'{N_SHAPES} shapes × {K} variantes = {N_SHAPES * K} series totales')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, (nombre, fn) in enumerate(SHAPES.items()):
    np.random.seed(i)
    ax = axes[i]
    ax.plot(fn(), 'o-', linewidth=1.5, markersize=3)
    ax.set_title(nombre, fontsize=10)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)

fig.suptitle('Shapes base (escala=1)', fontsize=12)
plt.tight_layout()
plt.show()

# 2. Generar dataset sintético con escalas y perturbaciones

In [ ]:
def agregar_perturbaciones(serie, prob_inicio_tardio=0.4, prob_cero=0.1, rng=None):
    """
    - arranque tardío: primeros N meses → -1  (prob_inicio_tardio)
    - ceros aleatorios en el medio              (prob_cero por mes)
    Devuelve serie modificada y mes de primer dato real.
    """
    if rng is None:
        rng = np.random.default_rng()
    s = serie.copy()
    primer_mes = 0

    if rng.random() < prob_inicio_tardio:
        inicio = rng.integers(1, T // 3)  # arranca entre mes 1 y mes 12
        s[:inicio] = -1.0
        primer_mes = int(inicio)

    # ceros aleatorios (solo donde ya existe el producto)
    for j in range(primer_mes, T):
        if rng.random() < prob_cero:
            s[j] = 0.0

    return s, primer_mes


# Generar todo el dataset
registros = []
rng = np.random.default_rng(99)

for shape_idx, (nombre_shape, fn) in enumerate(SHAPES.items()):
    for k in range(K):
        np.random.seed(shape_idx * 100 + k)
        serie_base = np.maximum(fn(), 0.0)  # sin negativos

        # escala log-uniforme [0.05, 200]
        escala_real = float(np.exp(rng.uniform(np.log(0.05), np.log(200))))
        serie_escalada = serie_base * escala_real

        serie_final, primer_mes = agregar_perturbaciones(
            serie_escalada, rng=rng
        )

        registros.append({
            'id':           f'{nombre_shape}_{k:02d}',
            'shape':        nombre_shape,
            'shape_idx':    shape_idx,
            'k':            k,
            'escala_real':  escala_real,
            'primer_mes':   primer_mes,
            'serie':        serie_final,        # array de T elementos
            'serie_base':   serie_base,         # shape pura, sin escala ni perturb
        })

print(f'{len(registros)} series generadas')
print(f'Rango escalas: {min(r["escala_real"] for r in registros):.3f} – {max(r["escala_real"] for r in registros):.1f}')
print(f'Series con arranque tardío: {sum(r["primer_mes"] > 0 for r in registros)}')

In [ ]:
# Visualizar 3 variantes del mismo shape para ver la dispersión de escalas
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, nombre_shape in enumerate(SHAPES):
    ax = axes[i]
    variantes = [r for r in registros if r['shape'] == nombre_shape][:4]
    for r in variantes:
        s = r['serie'].copy()
        s_plot = np.where(s == -1, np.nan, s)
        ax.plot(s_plot, alpha=0.7, linewidth=1.2, label=f'×{r["escala_real"]:.1f}')
    ax.set_title(nombre_shape, fontsize=9)
    ax.legend(fontsize=6)
    ax.grid(alpha=0.3)

fig.suptitle('4 variantes por shape — distinta escala + perturbaciones', fontsize=11)
plt.tight_layout()
plt.show()

# 3. Normalizaciones

Reglas comunes:
- `-1` = producto no existía → **no se toca**, se mantiene como sentinela
- `0` = existía pero no vendió → incluido en el denominador (es un valor real)
- La escala se calcula **solo sobre valores ≥ 0**

In [ ]:
def norm_sin(serie):
    """Sin normalizar — baseline."""
    return serie.copy(), 1.0


def norm_max(serie):
    """Divide por el máximo de valores >= 0. -1 se preserva."""
    reales = serie[serie >= 0]
    m = float(reales.max()) if len(reales) > 0 and reales.max() > 0 else 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / m
    return n, m


def norm_l2(serie):
    """Divide por norma L2 de valores >= 0. -1 se preserva."""
    reales = serie[serie >= 0]
    norma = float(np.sqrt((reales ** 2).sum()))
    if norma == 0:
        norma = 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / norma
    return n, norma


def norm_index(serie, n_base=3):
    """Divide por la media de los primeros n_base valores > 0. -1 se preserva."""
    positivos = serie[serie > 0]
    base = float(positivos[:n_base].mean()) if len(positivos) >= 1 else 1.0
    if base == 0:
        base = 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / base
    return n, base


def norm_reciente(serie, ventana=3):
    """Divide por la media de los últimos `ventana` valores >= 0. -1 se preserva."""
    reales = serie[serie >= 0]
    ultimos = reales[-ventana:] if len(reales) >= 1 else np.array([1.0])
    base = float(ultimos.mean()) if ultimos.mean() > 0 else 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / base
    return n, base


def norm_std(serie):
    """(serie - media) / std sobre valores >= 0, luego desplaza para que mínimo >= 0."""
    reales = serie[serie >= 0]
    mu = float(reales.mean()) if len(reales) > 0 else 0.0
    sigma = float(reales.std()) if len(reales) > 1 else 1.0
    if sigma == 0:
        sigma = 1.0
    n = serie.copy()
    n[serie >= 0] = (serie[serie >= 0] - mu) / sigma
    # desplazar para que el mínimo sea 0
    min_val = n[n >= -0.5].min() if any(n >= -0.5) else 0.0  # ignora -1 sentinelas
    if min_val < 0:
        n[serie >= 0] -= min_val
    # escala guardada = (mu, sigma) como tupla — desnormalizar: pred * sigma + mu
    return n, (mu, sigma)


NORMALIZACIONES = {
    'sin_norm': norm_sin,
    'max':      norm_max,
    'l2':       norm_l2,
    'index':    norm_index,
    'reciente': norm_reciente,
    'std':      norm_std,
}

print('Normalizaciones:', list(NORMALIZACIONES.keys()))

In [ ]:
# Verificar: 4 variantes del shape 'estacional' normalizadas
nombre_test = 'estacional'
variantes_test = [r for r in registros if r['shape'] == nombre_test][:4]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (nombre_norm, fn) in enumerate(NORMALIZACIONES.items()):
    ax = axes[i]
    for r in variantes_test:
        s_norm, escala = fn(r['serie'])
        s_plot = np.where(s_norm == -1, np.nan, s_norm)
        ax.plot(s_plot, alpha=0.8, linewidth=1.5)
    ax.set_title(f'norm={nombre_norm}', fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle(f'Shape "{nombre_test}" — 4 variantes escaladas, cada normalización', fontsize=11)
plt.tight_layout()
plt.show()
print('Si la normalización funciona, las 4 líneas deberían superponerse.')

# 4. Similaridad — distancia coseno intra-shape vs inter-shape

In [ ]:
def preparar_vector(serie):
    """Reemplaza -1 por 0 para calcular distancias."""
    v = serie.copy()
    v[v == -1] = 0.0
    return v


def calcular_distancias(registros, norm_fn):
    """Calcula todas las distancias coseno entre pares de series normalizadas."""
    vectores = []
    labels   = []
    for r in registros:
        v_norm, _ = norm_fn(r['serie'])
        vectores.append(preparar_vector(v_norm))
        labels.append(r['shape_idx'])

    n = len(vectores)
    intra, inter = [], []

    for i in range(n):
        for j in range(i + 1, n):
            d = cosine(vectores[i], vectores[j])
            if labels[i] == labels[j]:
                intra.append(d)
            else:
                inter.append(d)

    return np.array(intra), np.array(inter)


resultados_dist = {}
for nombre_norm, fn in NORMALIZACIONES.items():
    intra, inter = calcular_distancias(registros, fn)
    resultados_dist[nombre_norm] = {'intra': intra, 'inter': inter}
    sep = inter.mean() - intra.mean()
    print(f'{nombre_norm:12s}  intra={intra.mean():.4f}  inter={inter.mean():.4f}  separación={sep:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (nombre_norm, res) in enumerate(resultados_dist.items()):
    ax = axes[i]
    ax.hist(res['intra'], bins=40, alpha=0.6, color='steelblue', label='intra-shape (mismo)')
    ax.hist(res['inter'], bins=40, alpha=0.6, color='tomato',    label='inter-shape (distinto)')
    sep = res['inter'].mean() - res['intra'].mean()
    ax.set_title(f'{nombre_norm}  |  separación={sep:.3f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('distancia coseno')
    ax.grid(alpha=0.3)

fig.suptitle('Distribución de distancias coseno — cuanto más separadas, mejor normalización', fontsize=11)
plt.tight_layout()
plt.show()
print('Mayor separación = la normalización hace que el mismo shape quede más junto y distintos shapes más lejos.')

# 5. Clustering jerárquico — dendrograma

In [ ]:
from sklearn.metrics import adjusted_rand_score

def ari_clustering(registros, norm_fn, n_clusters=None):
    """Adjusted Rand Index del clustering jerárquico vs labels reales."""
    if n_clusters is None:
        n_clusters = N_SHAPES

    vectores = []
    labels_reales = []
    for r in registros:
        v_norm, _ = norm_fn(r['serie'])
        vectores.append(preparar_vector(v_norm))
        labels_reales.append(r['shape_idx'])

    Z = linkage(vectores, method='ward')
    labels_pred = fcluster(Z, n_clusters, criterion='maxclust')
    ari = adjusted_rand_score(labels_reales, labels_pred)
    return ari, Z, labels_reales


print('Adjusted Rand Index (1.0 = clustering perfecto, 0.0 = aleatorio):')
print()
ari_resultados = {}
for nombre_norm, fn in NORMALIZACIONES.items():
    ari, Z, labels_reales = ari_clustering(registros, fn)
    ari_resultados[nombre_norm] = (ari, Z)
    print(f'  {nombre_norm:12s}: ARI = {ari:.4f}')

In [ ]:
# Dendrograma de la mejor y peor normalización
sorted_ari = sorted(ari_resultados.items(), key=lambda x: x[1][0], reverse=True)
mejor_nombre, (mejor_ari, mejor_Z) = sorted_ari[0]
peor_nombre,  (peor_ari,  peor_Z)  = sorted_ari[-1]

# colores por shape
COLORES_SHAPE = plt.cm.tab10(np.linspace(0, 1, N_SHAPES))
shape_nombres = list(SHAPES.keys())
labels_plot = [f"{r['shape'][:4]}_{r['k']:02d}" for r in registros]
leaf_colors = [COLORES_SHAPE[r['shape_idx']] for r in registros]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, nombre, (ari, Z) in [
    (axes[0], mejor_nombre, (mejor_ari, mejor_Z)),
    (axes[1], peor_nombre,  (peor_ari,  peor_Z)),
]:
    dn = dendrogram(
        Z,
        labels=labels_plot,
        ax=ax,
        leaf_rotation=90,
        leaf_font_size=6,
        color_threshold=0,
    )
    # colorear labels por shape
    xlbls = ax.get_xmajorticklabels()
    for lbl in xlbls:
        shape_k = lbl.get_text()  # 'shap_01'
        idx = next((i for i, r in enumerate(registros) if f"{r['shape'][:4]}_{r['k']:02d}" == shape_k), 0)
        lbl.set_color(COLORES_SHAPE[registros[idx]['shape_idx']])

    ax.set_title(f'norm={nombre}  |  ARI={ari:.4f}', fontsize=10)
    ax.set_ylabel('distancia Ward')
    ax.grid(axis='y', alpha=0.3)

# leyenda de colores
handles = [plt.Line2D([0],[0], color=COLORES_SHAPE[i], linewidth=3, label=s) for i, s in enumerate(shape_nombres)]
axes[0].legend(handles=handles, fontsize=7, loc='upper right')

fig.suptitle('Dendrograma — mejor vs peor normalización\n(colores = shape real)', fontsize=11)
plt.tight_layout()
plt.show()

# 6. Árbol de decisión — ¿puede clasificar shapes con features normalizadas?

In [ ]:
from sklearn.model_selection import cross_val_score

print('Accuracy 5-fold CV de árbol de decisión (max_depth=5) para clasificar shape:')
print()

labels_reales = np.array([r['shape_idx'] for r in registros])
arbol_resultados = {}

for nombre_norm, fn in NORMALIZACIONES.items():
    X = []
    for r in registros:
        v_norm, _ = fn(r['serie'])
        X.append(preparar_vector(v_norm))
    X = np.array(X)

    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    scores = cross_val_score(clf, X, labels_reales, cv=5, scoring='accuracy')
    arbol_resultados[nombre_norm] = scores
    print(f'  {nombre_norm:12s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Visualizar árbol con la mejor normalización
mejor_norm_arbol = max(arbol_resultados, key=lambda k: arbol_resultados[k].mean())
fn_mejor = NORMALIZACIONES[mejor_norm_arbol]

X = np.array([preparar_vector(fn_mejor(r['serie'])[0]) for r in registros])
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X, labels_reales)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    clf,
    ax=ax,
    class_names=shape_nombres,
    feature_names=[f't{i}' for i in range(T)],
    filled=True,
    fontsize=7,
    max_depth=3,
)
ax.set_title(f'Árbol de decisión — norm={mejor_norm_arbol} (mejor)', fontsize=11)
plt.tight_layout()
plt.show()

# 7. Desescalado — error al predecir t+2 normalizado y desnormalizar

Usamos regresión lineal sobre los últimos 6 meses normalizados, predecimos t+2 normalizado, desnormalizamos y comparamos con el valor real.

In [ ]:
def pred_reg_norm(serie_norm, ventana=6, horizonte=2):
    """OLS sobre últimos `ventana` valores >= 0 normalizados, predice t+horizonte."""
    reales_idx = np.where(serie_norm[:-horizonte] >= 0)[0]
    if len(reales_idx) < 2:
        validos = serie_norm[serie_norm >= 0]
        return float(validos.mean()) if len(validos) > 0 else 0.0

    idx_use = reales_idx[-ventana:]
    y = serie_norm[idx_use]
    x = np.arange(len(y)).reshape(-1, 1)
    m = LinearRegression().fit(x, y)
    pred = float(m.predict([[len(y) - 1 + horizonte]])[0])
    return max(pred, 0.0)


def desescalar(pred_norm, escala, norm_nombre):
    """Desnormaliza la predicción."""
    if norm_nombre == 'std':
        mu, sigma = escala
        return max(pred_norm * sigma + mu, 0.0)
    else:
        return max(pred_norm * escala, 0.0)


# Evaluación: usar primeros T-2 meses como historia, predecir mes T-1 (t+2 desde T-3)
VENTANA_PRED = 6

print('RMSE del desescalado (predecir mes T usando reg_lineal sobre normalizado):')
print()

rmse_desescalado = {}

for nombre_norm, fn in NORMALIZACIONES.items():
    errores = []
    for r in registros:
        historia = r['serie'][:-HORIZONTE]
        valor_real = r['serie'][-1]  # mes T

        if valor_real < 0:  # si el producto no existe en el target, skip
            continue

        serie_norm, escala = fn(historia)
        pred_norm = pred_reg_norm(serie_norm, ventana=VENTANA_PRED, horizonte=HORIZONTE)
        pred_real = desescalar(pred_norm, escala, nombre_norm)

        errores.append((pred_real - valor_real) ** 2)

    rmse = float(np.sqrt(np.mean(errores)))
    rmse_desescalado[nombre_norm] = rmse
    print(f'  {nombre_norm:12s}: RMSE = {rmse:.4f}')

In [ ]:
# Visualizar predicciones desescaladas para un shape
nombre_shape_vis = 'estacional'
variantes_vis = [r for r in registros if r['shape'] == nombre_shape_vis][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, r in zip(axes, variantes_vis):
    historia = r['serie'][:-HORIZONTE]
    valor_real = float(r['serie'][-1])
    t_hist = np.arange(len(historia))
    s_plot = np.where(historia == -1, np.nan, historia)

    ax.plot(t_hist, s_plot, 'o-', color='steelblue', markersize=3, linewidth=1.5, label='historia')
    ax.scatter([T - 1], [valor_real], color='black', s=80, zorder=6, label=f'real={valor_real:.2f}')

    colores_pred = ['tomato', 'green', 'purple', 'orange', 'brown', 'gray']
    for (nombre_norm, fn), color in zip(NORMALIZACIONES.items(), colores_pred):
        serie_norm, escala = fn(historia)
        pred_norm = pred_reg_norm(serie_norm, ventana=VENTANA_PRED, horizonte=HORIZONTE)
        pred_real = desescalar(pred_norm, escala, nombre_norm)
        ax.scatter([T - 1], [pred_real], color=color, s=50, zorder=5, marker='D',
                   label=f'{nombre_norm}={pred_real:.2f}')

    ax.set_title(f'escala real = {r["escala_real"]:.2f}', fontsize=8)
    ax.legend(fontsize=5)
    ax.grid(alpha=0.3)

fig.suptitle(f'Predicciones desescaladas — shape "{nombre_shape_vis}"\n'
             f'(negro=real, resto=predicciones por normalización)', fontsize=10)
plt.tight_layout()
plt.show()

# 8. Resumen comparativo

In [ ]:
print('=' * 65)
print(f'  {"NORM":12s}  {"Separación":>11s}  {"ARI Cluster":>11s}  {"Acc Árbol":>10s}  {"RMSE Pred":>10s}')
print('=' * 65)

for nombre_norm in NORMALIZACIONES:
    res = resultados_dist[nombre_norm]
    sep = res['inter'].mean() - res['intra'].mean()
    ari = ari_resultados[nombre_norm][0]
    acc = arbol_resultados[nombre_norm].mean()
    rmse = rmse_desescalado[nombre_norm]
    print(f'  {nombre_norm:12s}  {sep:>11.4f}  {ari:>11.4f}  {acc:>10.4f}  {rmse:>10.4f}')

print('=' * 65)
print()
print('Separación coseno: mayor = mismo shape queda más junto')
print('ARI:               1.0 = clustering perfecto')
print('Acc árbol:         fracción de shapes clasificados correctamente')
print('RMSE pred:         error al predecir t+2 (incluye error de desescalado)')

In [ ]:
# Radar / bar chart comparativo
metricas = {
    'Separación\ncoseno':   {n: (resultados_dist[n]['inter'].mean() - resultados_dist[n]['intra'].mean()) for n in NORMALIZACIONES},
    'ARI\ncluster':        {n: ari_resultados[n][0] for n in NORMALIZACIONES},
    'Accuracy\nárbol':     {n: arbol_resultados[n].mean() for n in NORMALIZACIONES},
    'RMSE pred\n(invertido)': {n: 1 / (1 + rmse_desescalado[n]) for n in NORMALIZACIONES},
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colores = plt.cm.Set2(np.linspace(0, 1, len(NORMALIZACIONES)))
norms_list = list(NORMALIZACIONES.keys())

for ax, (titulo, vals) in zip(axes, metricas.items()):
    valores = [vals[n] for n in norms_list]
    bars = ax.bar(norms_list, valores, color=colores, edgecolor='white')
    ax.set_title(titulo, fontsize=10)
    ax.set_xticklabels(norms_list, rotation=30, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('Comparación de normalizaciones — mundo ideal escalado', fontsize=12)
plt.tight_layout()
plt.show()

# 9. Análisis de fallas — ¿cuándo falla cada normalización?

Miramos los casos donde la normalización más confunde las series.

In [ ]:
# Para cada normalización, encontrar los pares intra-shape con mayor distancia
# (series del mismo shape que quedan lejos después de normalizar)

mejor_norm = max(ari_resultados, key=lambda k: ari_resultados[k][0])
peor_norm  = min(ari_resultados, key=lambda k: ari_resultados[k][0])

print(f'Mejor normalización: {mejor_norm}  (ARI={ari_resultados[mejor_norm][0]:.4f})')
print(f'Peor normalización:  {peor_norm}   (ARI={ari_resultados[peor_norm][0]:.4f})')
print()

# Analizar fallas de la peor
fn_peor = NORMALIZACIONES[peor_norm]
pares_falla = []

for i, r1 in enumerate(registros):
    for j, r2 in enumerate(registros):
        if j <= i:
            continue
        if r1['shape'] != r2['shape']:
            continue
        v1, _ = fn_peor(r1['serie'])
        v2, _ = fn_peor(r2['serie'])
        d = cosine(preparar_vector(v1), preparar_vector(v2))
        pares_falla.append((d, r1, r2))

pares_falla.sort(key=lambda x: -x[0])
top_fallas = pares_falla[:4]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, (dist, r1, r2) in enumerate(top_fallas):
    # original
    ax = axes[0, col]
    for r, color in [(r1, 'steelblue'), (r2, 'tomato')]:
        s = np.where(r['serie'] == -1, np.nan, r['serie'])
        ax.plot(s, color=color, alpha=0.8, linewidth=1.5)
    ax.set_title(f'{r1["shape"]}\nescalas: {r1["escala_real"]:.1f} vs {r2["escala_real"]:.1f}', fontsize=8)
    ax.set_ylabel('original', fontsize=7)
    ax.grid(alpha=0.3)

    # normalizado
    ax = axes[1, col]
    for r, color in [(r1, 'steelblue'), (r2, 'tomato')]:
        v_norm, _ = fn_peor(r['serie'])
        s = np.where(v_norm == -1, np.nan, v_norm)
        ax.plot(s, color=color, alpha=0.8, linewidth=1.5)
    ax.set_title(f'dist coseno = {dist:.4f}', fontsize=8)
    ax.set_ylabel(f'norm={peor_norm}', fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle(f'Top 4 fallas de "{peor_norm}" — mismo shape, mayor distancia coseno', fontsize=10)
plt.tight_layout()
plt.show()

# 10. ¿Qué pasa con el spike?

El shape `spike` es el caso más difícil: un valor extremo domina la escala en `norm_max`.

In [ ]:
spike_records = [r for r in registros if r['shape'] == 'spike'][:4]

fig, axes = plt.subplots(len(NORMALIZACIONES), len(spike_records),
                          figsize=(14, 3 * len(NORMALIZACIONES)))

for row, (nombre_norm, fn) in enumerate(NORMALIZACIONES.items()):
    for col, r in enumerate(spike_records):
        ax = axes[row, col]
        v_norm, escala = fn(r['serie'])
        s = np.where(v_norm == -1, np.nan, v_norm)
        ax.plot(s, 'o-', markersize=2, linewidth=1)
        if col == 0:
            ax.set_ylabel(nombre_norm, fontsize=8)
        if row == 0:
            ax.set_title(f'escala={r["escala_real"]:.1f}', fontsize=8)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=6)

fig.suptitle('Shape "spike" — cómo lo trata cada normalización', fontsize=11)
plt.tight_layout()
plt.show()
print('norm_max: el spike siempre vale 1.0, el resto queda comprimido.')
print('norm_l2: el spike domina el denominador, efecto similar a max.')
print('norm_index: el spike NO afecta la escala si ocurre después de los primeros 3 meses.')
print('norm_reciente: el spike NO afecta la escala si no está en los últimos 3 meses.')